# 18 · Resource-Aware Distributed Tuning

A hyperparameter search explores many configurations, and scheduling every
trial the same way wastes hardware: a lightweight model leaves a whole GPU
mostly idle, a compute-heavy one is slowed down when it has to share one.
WarpRec gives each model the resources that fit its computational profile,
declared in two lines of its configuration. This guide runs the same searches
on one machine with two GPUs, changing only those lines, and measures what
each setting costs and buys. It covers:

- what Ray sees on the machine, and how many trials WarpRec runs at once for
  a given `cpu_per_trial` and `gpu_per_trial`;
- one GPU against two, and two trials packed on each GPU
  (`gpu_per_trial: 0.5`), for a lightweight model (BPR) and a compute-heavy
  one (LightGCN);
- pruning the weaker trials with ASHA, on top of packing;
- one trial trained on two GPUs with DDP (`gpu_per_trial: 2`), for each model;
- the energy and carbon of every run, measured with CodeCarbon.

Guide 10 covers search spaces, strategies and schedulers, guide 11 the swarm
pipeline and the dashboards, guide 15 moving a model from the design pipeline
to Ray.

**Requirements:** a Linux machine with two CUDA GPUs (this notebook was run on
two RTX 4090 and 32 CPU cores) and `pip install "warprec[dashboard]" jupyter`.
Runs in about 25 minutes. MovieLens-100K is small for these GPUs: even
LightGCN, given large embeddings and four layers of propagation, leaves them
mostly idle. The numbers show how to measure what each setting buys, and why
the answer depends on how much work a trial gives its GPU.

In [1]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()  # downloads and converts it as guide 1 shows
RUNS = Path("../runs/18-resource-aware-distributed-tuning")
shutil.rmtree(RUNS, ignore_errors=True)
RUNS.mkdir(parents=True)
logger.disable("warprec")

## 1. The machine and the trials it holds

Ray sees every GPU the process may use. `CUDA_VISIBLE_DEVICES` restricts that,
which is how the runs below compare one GPU with two on the same machine.

In [2]:
import ray

ray.init(logging_level="ERROR", include_dashboard=False)
node = ray.nodes()[0]["Resources"]
print({key: node[key] for key in ("CPU", "GPU")})

{'CPU': 32.0, 'GPU': 2.0}


A trial is a Ray Train run: one worker, or one per GPU when `gpu_per_trial` is
an integer above 1, which then trains with DDP. Unless
`max_concurrent_trials` is set, WarpRec runs as many trials at once as the
nodes can host, counting each worker against the node's scarcest resource
(guide 10). Every configuration here gives each worker 4 CPUs, which size its
data loader (3 processes): 4 per trial, 8 for a trial on two GPUs. With 32
cores the GPUs are the scarcer resource, so a whole GPU per trial fits two
trials at once, half a GPU fits four, and a trial on two GPUs runs alone:

In [3]:
from warprec.recommenders.trainer import Trainer
from warprec.recommenders.trainer.trainer import trials_that_fit
from warprec.utils.config.config import load_train_configuration

trainer = Trainer(storage_path=str(RUNS / "probe"))
footprints = {}
for name in ["bpr", "bpr-packed", "lightgcn", "lightgcn-ddp"]:
    config = load_train_configuration(f"configs/{name}.yml")
    optimization = next(iter(config.models.values()))["optimization"]
    # The footprint the trainer computes for a trial of this configuration
    scaling = trainer._get_scaling_config(
        optimization["cpu_per_trial"], optimization["gpu_per_trial"], device="cuda"
    )
    footprints[name] = {
        "gpu_per_trial": optimization["gpu_per_trial"],
        "workers per trial": scaling["num_workers"],
        "each worker": scaling["resources_per_worker"],
        "trials at once": trials_that_fit(ray.nodes(), scaling),
    }
ray.shutdown()
pd.DataFrame(footprints).T

,gpu_per_trial,workers per trial,each worker,trials at once
bpr,1.0,1,"{'CPU': 4.0, 'GPU': 1.0}",2
bpr-packed,0.5,1,"{'CPU': 4.0, 'GPU': 0.5}",4
lightgcn,1.0,1,"{'CPU': 4.0, 'GPU': 1.0}",2
lightgcn-ddp,2.0,2,"{'CPU': 4.0, 'GPU': 1.0}",1


## 2. One GPU, two GPUs, two trials per GPU

Each search is an 8-trial grid. BPR scores a user against an item with an
embedding lookup: a batch keeps a GPU busy for a fraction of a millisecond,
and most of an epoch goes into sampling negatives on the CPU. LightGCN
propagates the embeddings of the whole user-item graph at every step, here
512 or 1024 dimensions through 3 or 4 layers, which makes it the heavier of
the two.

In [4]:
print(Path("configs/bpr.yml").read_text())

# An 8-trial grid of BPR, one trial per GPU.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: bpr
  writing_method: local
  local_experiment_path: ../runs/18-resource-aware-distributed-tuning/
models:
  BPR:                        # light: an embedding lookup per sample
    optimization:
      cpu_per_trial: 4
      gpu_per_trial: 1
      properties:
        seed: 42
    embedding_size: [grid, 64, 128]
    learning_rate: [grid, 0.001, 0.005]
    reg_weight: [grid, 0.00001, 0.0001]
    neg_samples: 4
    batch_size: 1024
    epochs: 20
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall]
  validation_metric: nDCG@10
general:
  device: cuda
  ray_address: local          # a private Ray instance with every GPU it can see
  r

The packed configuration differs in one line, and LightGCN's are built the
same way:

In [5]:
import difflib


def changed(a, b):
    """The lines of configs/b.yml that differ from configs/a.yml."""
    lines = difflib.unified_diff(
        Path(f"configs/{a}.yml").read_text().splitlines(True),
        Path(f"configs/{b}.yml").read_text().splitlines(True),
        f"{a}.yml",
        f"{b}.yml",
        n=0,
    )
    print("".join(line for line in lines if not line.startswith("@@")))


changed("bpr", "bpr-packed")

--- bpr.yml
+++ bpr-packed.yml
-# An 8-trial grid of BPR, one trial per GPU.
+# bpr.yml with two trials sharing each GPU.
-  dataset_name: bpr
+  dataset_name: bpr-packed
-      gpu_per_trial: 1
+      gpu_per_trial: 0.5            # two trials per GPU



A helper runs a configuration as `python -m warprec.run` would, on the GPUs it
is given, and reads back what the run wrote: the time report (guide 11), the
test results and the rows CodeCarbon appended to `emissions.csv`.

MovieLens-100K epochs last a fraction of a second, shorter than the 2 seconds
at which Ray Train collects a trial's reports by default: a trial reports
every epoch and cannot report again before the last report is collected, so
every epoch would last 2 seconds whatever the model. The runs here ask Ray
Train to collect them every 0.1 seconds instead, through
`RAY_TRAIN_HEALTH_CHECK_INTERVAL_S`. Epochs of a few seconds or more, as on
MovieLens-1M and beyond, do not need it.

In [6]:
def run(name, gpus="0,1", pipeline="train"):
    """Run configs/<name>.yml on the GPUs given; return what it measured."""
    out = RUNS / name
    shutil.rmtree(out, ignore_errors=True)
    env = {
        **os.environ,
        "CUDA_VISIBLE_DEVICES": gpus,
        "RAY_TRAIN_HEALTH_CHECK_INTERVAL_S": "0.1",  # see above
    }
    start = time.perf_counter()
    with open(RUNS / f"{name}-gpus{gpus.replace(',', '')}.log", "w") as log:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "warprec.run",
                "-c",
                f"configs/{name}.yml",
                "-p",
                pipeline,
            ],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=True,
        )
    wall = time.perf_counter() - start
    report = pd.read_csv(next((out / "evaluation").glob("Time_Report*")), sep="\t")
    results = pd.read_csv(next((out / "evaluation").glob("Overall_Results*")), sep="\t")
    emissions = pd.read_csv(out / "ray_results" / "emissions.csv")
    summary = {
        "run (s)": wall,
        "search (s)": report["Hyperparameter Exploration Time"].sum(),
        "mean trial (s)": report["Average Trial Time (s)"].mean(),
        "nDCG@10": results.set_index("Model")["nDCG"].to_dict(),
        "kWh": emissions["energy_consumed"].sum(),
        "kg CO2eq": emissions["emissions"].sum(),
    }
    return summary, report


setups = {
    "1 GPU": ("", "0"),
    "2 GPUs": ("", "0,1"),
    "2 GPUs, 2 trials per GPU": ("-packed", "0,1"),
}
packing = {}
for model in ["bpr", "lightgcn"]:
    for setup, (suffix, gpus) in setups.items():
        summary, _ = run(model + suffix, gpus)
        packing[(model, setup)] = summary

table = pd.DataFrame(packing).T
table["nDCG@10"] = [next(iter(v.values())) for v in table["nDCG@10"]]
table["speed-up"] = [
    table.loc[(m, "1 GPU"), "search (s)"] / row["search (s)"]
    for (m, _), row in table.iterrows()
]
table.astype(float).round(
    {
        "run (s)": 0,
        "search (s)": 0,
        "mean trial (s)": 1,
        "nDCG@10": 4,
        "kWh": 4,
        "kg CO2eq": 4,
        "speed-up": 2,
    }
)

run (s)  search (s)  mean trial (s)  \
bpr      1 GPU                       292.0       272.0            29.9   
         2 GPUs                      165.0       145.0            32.2   
         2 GPUs, 2 trials per GPU    102.0        79.0            34.7   
lightgcn 1 GPU                       177.0       157.0            15.8   
         2 GPUs                      107.0        84.0            16.8   
         2 GPUs, 2 trials per GPU     69.0        46.0            18.4   

                                   nDCG@10     kWh  kg CO2eq  speed-up  
bpr      1 GPU                      0.1112  0.0095    0.0031      1.00  
         2 GPUs                     0.1112  0.0065    0.0022      1.87  
         2 GPUs, 2 trials per GPU   0.1112  0.0038    0.0013      3.45  
lightgcn 1 GPU                      0.1117  0.0058    0.0019      1.00  
         2 GPUs                     0.1117  0.0042    0.0014      1.87  
         2 GPUs, 2 trials per GPU   0.1117  0.0029    0.0010      3.42

`search (s)` is the search's own wall clock, from the time report; `run (s)`
adds starting Ray, preparing the data and evaluating the best model on the
test set. Every setup trains the same 8 configurations with the same seeds, so
the test nDCG@10 of the best one does not depend on where it ran.

A second GPU runs twice the trials at once and nearly halves both searches
(1.87×). Packing two trials on each GPU doubles that again, to 3.45× for BPR
and 3.42× for LightGCN: each trial gets about 16% slower (BPR from 29.9 to
34.7 seconds, LightGCN from 15.8 to 18.4), far less than the trials gained.
`mean trial (s)` is the column to watch. On MovieLens-100K neither model keeps
a 4090 busy: LightGCN's largest configuration trains an epoch in 0.38
seconds, and about half of a 16-second trial goes into starting the worker
and validating. A GPU that is mostly idle has room for a second trial, so
packing pays off for both. A model that already keeps its GPU busy behaves
differently: two trials on one GPU each take about twice as long, packing buys
no throughput, and such a model is better off with a GPU of its own. Where
that happens depends on the model, the data and the GPU, so it is worth
measuring with a short search before a long one, as here.

The energy follows the wall clock: CodeCarbon measures the whole machine, idle
GPUs included, so the packed BPR search used 0.0038 kWh against 0.0095 on one
GPU.

## 3. Pruning with ASHA

Packing makes more trials run at once; a scheduler makes them shorter. ASHA
compares the trials at rungs (here after 4, 8 and 16 epochs) and stops the
weaker ones, freeing their GPU share for the next trial.

In [7]:
changed("bpr-packed", "bpr-asha")

--- bpr-packed.yml
+++ bpr-asha.yml
-# bpr.yml with two trials sharing each GPU.
+# bpr-packed.yml with ASHA stopping the weaker trials early.
-  dataset_name: bpr-packed
+  dataset_name: bpr-asha
-      gpu_per_trial: 0.5            # two trials per GPU
+      gpu_per_trial: 0.5
+        max_t: 20                 # ASHA: the longest a trial runs, in epochs
+        grace_period: 4           # every trial runs at least 4
+        reduction_factor: 2       # half the trials stop at each rung
+      scheduler: asha



In [8]:
asha, _ = run("bpr-asha")


def epochs_per_trial(name):
    """How many epochs each trial of a search ran, from its Ray Tune results."""
    trials = (RUNS / name / "ray_results").glob("*__BPR*/*/progress.csv")
    return sorted(len(pd.read_csv(p)) for p in trials)


pd.DataFrame(
    {
        scheduler: {
            "search (s)": round(summary["search (s)"]),
            "epochs per trial": epochs_per_trial(name),
            "nDCG@10": round(summary["nDCG@10"]["BPR"], 4),
            "kWh": round(summary["kWh"], 4),
        }
        for scheduler, name, summary in [
            ("FIFO", "bpr-packed", packing[("bpr", "2 GPUs, 2 trials per GPU")]),
            ("ASHA", "bpr-asha", asha),
        ]
    }
).T

,search (s),epochs per trial,nDCG@10,kWh
FIFO,79,"[20, 20, 20, 20, 20, 20, 20, 20]",0.1112,0.0038
ASHA,66,"[4, 4, 4, 4, 8, 16, 16, 20]",0.1112,0.0028


ASHA stopped four trials after 4 epochs, one after 8 and two after 16, and let
one run all 20: 76 epochs instead of 160. The best configuration is among
those it kept, so the search ends on the same model (nDCG@10 0.1112) in 66
seconds instead of 79, for 0.0028 kWh instead of 0.0038. The time saved is
smaller than the epochs saved because the search lasts as long as its longest
trials, the promising ones ASHA keeps, and every trial still pays its start.

## 4. One trial on two GPUs

Packing and pruning scale a search horizontally, across trials. DDP scales a
single trial vertically: `gpu_per_trial: 2` trains it with two workers, one
per GPU. Each worker takes half of every epoch's batches, and the gradients
are averaged between the GPUs after every step. That synchronisation is the
price, and it pays off when a step computes enough to hide it. Each model
trains one configuration of its grid, on one GPU and then on two:

In [9]:
changed("lightgcn-single", "lightgcn-ddp")

--- lightgcn-single.yml
+++ lightgcn-ddp.yml
-# The same LightGCN trial on one GPU.
+# One LightGCN trial, trained on two GPUs with DDP.
-  dataset_name: lightgcn-single
+  dataset_name: lightgcn-ddp
-      cpu_per_trial: 4
-      gpu_per_trial: 1
+      cpu_per_trial: 8              # 4 for each of the two workers
+      gpu_per_trial: 2              # one trial, one worker on each of two GPUs



In [10]:
ddp = {}
for model in ["bpr", "lightgcn"]:
    single, single_report = run(f"{model}-single", "0")
    double, double_report = run(f"{model}-ddp", "0,1")
    one = single_report["Best Trial Mean Epoch Train Time (s)"].iloc[0]
    two = double_report["Best Trial Mean Epoch Train Time (s)"].iloc[0]
    ddp[model] = {
        "epoch, 1 GPU (s)": one,
        "epoch, 2 GPUs (s)": two,
        "speed-up": one / two,
        "efficiency": one / two / 2,
        "samples/s, 1 GPU": single_report[
            "Best Trial Train Throughput (samples/s)"
        ].iloc[0],
        "samples/s, 2 GPUs": double_report[
            "Best Trial Train Throughput (samples/s)"
        ].iloc[0],
        "run, 1 GPU (s)": single["run (s)"],
        "run, 2 GPUs (s)": double["run (s)"],
        "kWh, 1 GPU": single["kWh"],
        "kWh, 2 GPUs": double["kWh"],
        "nDCG@10, 1 GPU": next(iter(single["nDCG@10"].values())),
        "nDCG@10, 2 GPUs": next(iter(double["nDCG@10"].values())),
    }
pd.DataFrame(ddp).round(4)

,bpr,lightgcn
"epoch, 1 GPU (s)",1.1264,0.3800
"epoch, 2 GPUs (s)",0.6242,0.3125
speed-up,1.8047,1.2158
efficiency,0.9023,0.6079
"samples/s, 1 GPU",71187.8310,211024.3062
"samples/s, 2 GPUs",128471.7971,256565.4043
"run, 1 GPU (s)",56.3843,43.1287
"run, 2 GPUs (s)",47.5533,43.4596
"kWh, 1 GPU",0.0013,0.0010
"kWh, 2 GPUs",0.0013,0.0012


Two GPUs make a BPR epoch 1.80× faster (90% efficiency) and a LightGCN epoch
only 1.22× faster (61%), for different reasons:

- BPR's epoch is bound by the CPU that samples its negatives, not by the GPU.
  Each worker brings its own data loader, so two workers sample twice as fast,
  and the speed-up follows them.
- LightGCN's workers each still propagate the whole graph at every step: DDP
  splits the batches, not the graph. After every step the gradients of all
  the embeddings are averaged between the GPUs, and RTX 4090s have no NVLink,
  so the exchange goes through the PCIe bus. On a graph this small, a step
  computes too little to hide it.

Over a whole run of 20 epochs there is little left to save: BPR's run drops
from 56 to 48 seconds for the same energy, LightGCN's stays at 43 seconds and
the second GPU adds its energy (0.0010 against 0.0012 kWh). The models also
differ slightly (BPR 0.1112 against 0.1060): with two workers each step sees
twice the samples and an epoch has half the steps, which is a different
optimisation. DDP is for trials that are too slow or too large for one GPU,
where a step computes enough to hide the exchange; for trials of this size, a
second GPU does more as room for another trial.

## 5. Energy and carbon

Every run above measured itself: with `tracking_mode: machine`, one CodeCarbon
tracker per Ray node covers the whole run, the GPUs, the CPU and the memory,
idle or not. A setup that keeps the GPUs busy for less time therefore costs
less energy for the same search.

In [11]:
energy = {
    f"{m} · {s}": {"kWh": v["kWh"], "kg CO2eq": v["kg CO2eq"]}
    for (m, s), v in packing.items()
}
energy["bpr · ASHA"] = {"kWh": asha["kWh"], "kg CO2eq": asha["kg CO2eq"]}
pd.DataFrame(energy).T.round(4)

,kWh,kg CO2eq
bpr · 1 GPU,0.0095,0.0031
bpr · 2 GPUs,0.0065,0.0022
"bpr · 2 GPUs, 2 trials per GPU",0.0038,0.0013
lightgcn · 1 GPU,0.0058,0.0019
lightgcn · 2 GPUs,0.0042,0.0014
"lightgcn · 2 GPUs, 2 trials per GPU",0.0029,0.0010
bpr · ASHA,0.0028,0.0009


For the same search and the same best model, packing and pruning together
took BPR from 0.0095 kWh on one GPU to 0.0028, less than a third.

The [Dashboard configuration](https://warprec.readthedocs.io/en/latest/configuration/dashboard/)
page lists what CodeCarbon measures under each tracking mode, and where it
estimates instead.

## Summary

| Key | What it does | Docs |
|---|---|---|
| `optimization.cpu_per_trial`, `gpu_per_trial` | the resources of a trial; a fraction of a GPU packs trials, an integer above 1 trains one trial on that many GPUs with DDP | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `optimization.max_concurrent_trials` | caps the trials running at once; unset, WarpRec computes it from the nodes that can host a trial | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `optimization.scheduler: asha`, `properties.max_t`, `grace_period`, `reduction_factor` | stop the weaker trials at each rung | [Models](https://warprec.readthedocs.io/en/latest/configuration/models/) |
| `general.device: cuda` | train and evaluate on the GPUs Ray assigns | [General](https://warprec.readthedocs.io/en/latest/configuration/general/) |
| `dashboard.codecarbon` | energy and carbon of every run, in `ray_results/emissions.csv` | [Dashboard](https://warprec.readthedocs.io/en/latest/configuration/dashboard/) |